In [1]:
from langchain_core.runnables.graph import MermaidDrawMethod



In [1]:
# 1. Basic imports and setup
import os
import sys
from dotenv import load_dotenv
from IPython.display import display

# Point to project root if notebook is nested
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

# Load any .env variables your nodes may need
load_dotenv(override=True)

# 2. Import graph tools
from langchain_core.runnables.graph import CurveStyle, NodeStyles, MermaidDrawMethod  # optional for fallback
from langgraph.graph import StateGraph, END

# 3. Build your workflow graph (attempt visualize_graph.py or fallback to nodes.py)
workflow = None
try:
    from visualize_graph import build_workflow
    workflow = build_workflow()
    print("✅ Imported build_workflow() from visualize_graph.py")
except Exception as e:
    print("ℹ️ Could not import build_workflow():", e)
    print("ℹ️ Falling back to nodes.py...")
    from nodes import (
        cache_check_node,
        retrieve_node,
        relevance_check_node,
        answer_node,
        summarize_history_node,
    )
    workflow = StateGraph(dict)
    workflow.add_node("cache_check", cache_check_node)
    workflow.add_node("retrieve", retrieve_node)
    workflow.add_node("relevance", relevance_check_node)
    workflow.add_node("answer", answer_node)
    workflow.add_node("summarize", summarize_history_node)
    workflow.set_entry_point("cache_check")
    workflow.add_edge("cache_check", "retrieve")
    workflow.add_edge("retrieve", "relevance")
    workflow.add_conditional_edges(
        "relevance",
        lambda s: "answer" if s.get("context_relevant") else "answer",
        {"answer": "answer"},
    )
    workflow.add_edge("answer", "summarize")
    workflow.add_edge("summarize", END)
    print("✅ Built workflow from nodes.py")

# 4. Compile graph into runnable/workable form
print("\nCompiling workflow...")
graph = workflow.compile()
print("✅ Graph compiled.")

# 5. Print ASCII visualization of the graph
ascii_diagram = graph.get_graph().draw_ascii()
print("\n--- Workflow Graph (ASCII) ---\n")
print(ascii_diagram)


✅ Imported build_workflow() from visualize_graph.py

Compiling workflow...
✅ Graph compiled.

--- Workflow Graph (ASCII) ---

 +-----------+   
 | __start__ |   
 +-----------+   
        *        
        *        
        *        
+-------------+  
| cache_check |  
+-------------+  
        *        
        *        
        *        
  +----------+   
  | retrieve |   
  +----------+   
        *        
        *        
        *        
 +-----------+   
 | relevance |   
 +-----------+   
        .        
        .        
        .        
  +--------+     
  | answer |     
  +--------+     
        *        
        *        
        *        
 +-----------+   
 | summarize |   
 +-----------+   
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


In [2]:
# 1. Basic imports and setup
import os
import sys
from dotenv import load_dotenv

# Adjust to your project root if needed
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

# Load environment variables if your workflow relies on them
load_dotenv(override=True)

# 2. Import necessary graph utilities
from langchain_core.runnables.graph import CurveStyle, NodeStyles
from langgraph.graph import StateGraph, END

# 3. Build your workflow graph (visualize_graph.py or fallback to nodes.py)
workflow = None
try:
    from visualize_graph import build_workflow
    workflow = build_workflow()
    print("✅ Imported build_workflow() from visualize_graph.py")
except Exception as e:
    print("ℹ️ Could not import build_workflow():", e)
    print("ℹ️ Falling back to nodes.py...")
    from nodes import (
        cache_check_node,
        retrieve_node,
        relevance_check_node,
        answer_node,
        summarize_history_node,
    )

    workflow = StateGraph(dict)
    workflow.add_node("cache_check", cache_check_node)
    workflow.add_node("retrieve", retrieve_node)
    workflow.add_node("relevance", relevance_check_node)
    workflow.add_node("answer", answer_node)
    workflow.add_node("summarize", summarize_history_node)
    workflow.set_entry_point("cache_check")
    workflow.add_edge("cache_check", "retrieve")
    workflow.add_edge("retrieve", "relevance")
    workflow.add_conditional_edges(
        "relevance",
        lambda s: "answer" if s.get("context_relevant") else "answer",
        {"answer": "answer"},
    )
    workflow.add_edge("answer", "summarize")
    workflow.add_edge("summarize", END)
    print("✅ Built workflow from nodes.py")

# 4. Compile graph
print("\nCompiling workflow...")
graph = workflow.compile()
print("✅ Graph compiled.")

# 5. Generate Mermaid code
mermaid_code = graph.get_graph().draw_mermaid(
    with_styles=True,                          # Include styling directives
    curve_style=CurveStyle.LINEAR,             # Customize edge curves
    node_colors=NodeStyles(),                  # Default node coloring
    wrap_label_n_words=9                       # Text wrapping for labels (default)
)

print("\n--- Mermaid Diagram Code ---\n")
print(mermaid_code)


✅ Imported build_workflow() from visualize_graph.py

Compiling workflow...
✅ Graph compiled.

--- Mermaid Diagram Code ---

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	cache_check(cache_check)
	retrieve(retrieve)
	relevance(relevance)
	answer(answer)
	summarize(summarize)
	__end__([<p>__end__</p>]):::last
	__start__ --> cache_check;
	answer --> summarize;
	cache_check --> retrieve;
	relevance -.-> answer;
	retrieve --> relevance;
	summarize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [2]:
# visualization_fallback.py

import os
import sys
from dotenv import load_dotenv
from typing_extensions import TypedDict
from typing import Annotated
import operator

# Setup imports
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)
load_dotenv(override=True)

from langgraph.graph import StateGraph, END
from nodes import (
    cache_check_node,
    retrieve_node,
    relevance_check_node,
    internet_search_node,
    answer_node,
    summarize_history_node,
)

# Define a state schema that supports merging
class VizState(TypedDict):
    __root__: Annotated[list, operator.add]

# Build the workflow
workflow = StateGraph(VizState)

for name, node in [
    ("cache_check", cache_check_node),
    ("retrieve", retrieve_node),
    ("relevance", relevance_check_node),
    ("internet_search", internet_search_node),
    ("answer", answer_node),
    ("summarize", summarize_history_node),
]:
    workflow.add_node(name, node)

workflow.set_entry_point("cache_check")
workflow.add_edge("cache_check", "retrieve")
workflow.add_edge("retrieve", "relevance")

workflow.add_conditional_edges(
    "relevance",
    lambda s: "answer" if s.get("context_relevant") else "internet_search",
    {"answer": "answer", "internet_search": "internet_search"},
)

workflow.add_edge("internet_search", "answer")
workflow.add_edge("answer", "summarize")
workflow.add_edge("summarize", END)

# Compile the graph
graph = workflow.compile()

# Render Mermaid
mermaid_code = graph.get_graph().draw_mermaid()
print("\n--- Mermaid Diagram Code ---\n")
print(mermaid_code)



--- Mermaid Diagram Code ---

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	cache_check(cache_check)
	retrieve(retrieve)
	relevance(relevance)
	internet_search(internet_search)
	answer(answer)
	summarize(summarize)
	__end__([<p>__end__</p>]):::last
	__start__ --> cache_check;
	answer --> summarize;
	cache_check --> retrieve;
	internet_search --> answer;
	relevance -.-> answer;
	relevance -.-> internet_search;
	retrieve --> relevance;
	summarize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

